# 09 — Model Evaluation + Selection

This notebook uses the **validation set** to compare Logistic Regression, Random Forest, and XGBoost.
The untouched test set is intentionally not used for model selection.

Checks included:
- PR-AUC
- ROC-AUC
- Precision / Recall / F1
- Balanced accuracy / MCC
- Brier score
- Grouped train-vs-validation diagnostics
- Confusion matrix
- Validation error analysis
- Permutation importance
- Missing-value and unknown-category robustness

In [1]:
# AI-Based IAM Permission Optimizer — Model V2
# Run notebooks in order: 01 → 10
# Raw CSVs should be available in the project root or adjust RAW_DIR below.
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    matthews_corrcoef, brier_score_loss, confusion_matrix, classification_report
)
from sklearn.inspection import permutation_importance
from IPython.display import display

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
OUTPUT_DIR = PROJECT_DIR / "outputs"

df = pd.read_csv(DATA_DIR / "labeled_dataset.csv")
splits = pd.read_csv(DATA_DIR / "splits.csv")
df["split"] = splits["split"].values

artifact_files = {
    "logistic_regression": ARTIFACT_DIR / "logistic_regression_candidate.joblib",
    "random_forest": ARTIFACT_DIR / "random_forest_candidate.joblib",
    "xgboost": ARTIFACT_DIR / "xgboost_candidate.joblib",
}

models = {}
for name, path in artifact_files.items():
    if path.exists():
        models[name] = joblib.load(path)

if not models:
    raise FileNotFoundError("No trained model artifacts found. Run 06, 07, and/or 08 first.")

features = models[next(iter(models))]["feature_columns"]
X = df[features].copy()
y = df["target"].astype(int)
groups = df["user_id"]

fit_mask = df["split"].eq("train")
val_mask = df["split"].eq("validation")
X_fit, y_fit, groups_fit = X.loc[fit_mask], y.loc[fit_mask], groups.loc[fit_mask]
X_val, y_val, groups_val = X.loc[val_mask], y.loc[val_mask], groups.loc[val_mask]


In [2]:
def metric_row(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision_excessive": precision_score(y_true, pred, zero_division=0),
        "recall_excessive": recall_score(y_true, pred, zero_division=0),
        "f1_excessive": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, prob),
        "pr_auc": average_precision_score(y_true, prob),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "mcc": matthews_corrcoef(y_true, pred),
        "brier_score": brier_score_loss(y_true, prob),
    }

rows = []
validation_probabilities = {}

for name, artifact in models.items():
    estimator = artifact["estimator"]
    prob = estimator.predict_proba(X_val)[:, 1]
    validation_probabilities[name] = prob
    row = metric_row(y_val, prob, threshold=0.5)
    row["model"] = name
    row["cv_pr_auc"] = artifact["best_cv_pr_auc"]
    rows.append(row)

comparison = pd.DataFrame(rows).set_index("model").sort_values("pr_auc", ascending=False)
display(comparison)
comparison.to_csv(OUTPUT_DIR / "model_validation_comparison.csv")

,accuracy,precision_excessive,recall_excessive,f1_excessive,roc_auc,pr_auc,balanced_accuracy,mcc,brier_score,cv_pr_auc
model,,,,,,,,,,
xgboost,0.97150,0.912260,0.94875,0.930147,0.992027,0.978441,0.962969,0.912531,0.027632,0.935935
random_forest,0.82050,0.539575,0.69875,0.608932,0.866776,0.704878,0.774844,0.501900,0.132020,0.655418
logistic_regression,0.76375,0.447119,0.76625,0.564717,0.840653,0.590624,0.764687,0.446138,0.164245,0.553074


In [3]:
# Select using validation PR-AUC only. Test data is not consulted.
selected_model_name = comparison.index[0]
selected_artifact = models[selected_model_name]
selected_estimator = selected_artifact["estimator"]
selected_prob = validation_probabilities[selected_model_name]

print("Selected candidate for finalization:", selected_model_name)
print("Selection metric: validation PR-AUC")

Selected candidate for finalization: xgboost
Selection metric: validation PR-AUC


In [4]:
# Tune the operating threshold on validation data.
def tune_threshold(y_true, score, minimum_recall=0.90):
    rows = []
    for threshold in np.round(np.linspace(0.05, 0.95, 181), 4):
        pred = (score >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0)
        })
    table = pd.DataFrame(rows)
    eligible = table[table["recall"] >= minimum_recall]
    chosen = (eligible if len(eligible) else table).sort_values(["f1", "precision"], ascending=False).iloc[0]
    return float(chosen["threshold"]), table

validation_threshold, threshold_table = tune_threshold(y_val.values, selected_prob, minimum_recall=0.90)
print("Validation-selected threshold:", validation_threshold)
threshold_table.to_csv(OUTPUT_DIR / "selected_model_thresholds_validation.csv", index=False)

Validation-selected threshold: 0.53


In [5]:
# Explicit train-vs-validation grouped CV diagnostic.
diagnostic_cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=44)
diagnostic = cross_validate(
    clone(selected_estimator),
    X_fit,
    y_fit,
    groups=groups_fit,
    cv=diagnostic_cv,
    scoring={"pr_auc": "average_precision", "f1": "f1", "precision": "precision", "recall": "recall"},
    return_train_score=True,
    n_jobs=1
)

diag_rows = []
for metric in ["pr_auc", "f1", "precision", "recall"]:
    train_mean = diagnostic[f"train_{metric}"].mean()
    valid_mean = diagnostic[f"test_{metric}"].mean()
    diag_rows.append({"metric": metric, "train_mean": train_mean, "validation_mean": valid_mean, "gap": train_mean - valid_mean})

diagnostics = pd.DataFrame(diag_rows).set_index("metric")
display(diagnostics)
diagnostics.to_csv(OUTPUT_DIR / "selected_model_overfit_diagnostics.csv")

,train_mean,validation_mean,gap
metric,,,
pr_auc,0.996515,0.935935,0.060581
f1,0.961945,0.864717,0.097228
precision,0.929326,0.838328,0.090997
recall,0.996944,0.893333,0.103611


In [6]:
# Validation confusion matrix and error analysis.
val_pred = (selected_prob >= validation_threshold).astype(int)
cm = confusion_matrix(y_val, val_pred)
print(classification_report(y_val, val_pred, target_names=["INTENDED", "EXCESSIVE"], zero_division=0))

errors = df.loc[val_mask, [
    "row_id", "user_id", "role_id", "action", "usage_count",
    "days_since_last_use", "risk_level", "risk_weight", "permission_status"
]].copy()
errors["risk_score"] = selected_prob
errors["predicted"] = np.where(val_pred == 1, "EXCESSIVE", "INTENDED")
errors["error_type"] = np.select(
    [(y_val.values == 1) & (val_pred == 0), (y_val.values == 0) & (val_pred == 1)],
    ["FALSE_NEGATIVE", "FALSE_POSITIVE"],
    default="CORRECT"
)

display(errors[errors["error_type"] != "CORRECT"].sort_values("risk_score", ascending=False).head(50))
errors.to_csv(OUTPUT_DIR / "validation_error_analysis.csv", index=False)

              precision    recall  f1-score   support

    INTENDED       0.99      0.98      0.98      3200
   EXCESSIVE       0.92      0.94      0.93       800

    accuracy                           0.97      4000
   macro avg       0.95      0.96      0.96      4000
weighted avg       0.97      0.97      0.97      4000



,row_id,user_id,role_id,action,usage_count,days_since_last_use,risk_level,risk_weight,permission_status,risk_score,predicted,error_type
2416,2417,synthetic-user-013,DataEngineer-Role,kinesis:DescribeLimits,0,78,LOW,2,INTENDED,0.931324,EXCESSIVE,FALSE_POSITIVE
7354,7355,synthetic-user-037,SecurityEngineer-Role,secretsmanager:DescribeSecret,1,31,LOW,2,INTENDED,0.887596,EXCESSIVE,FALSE_POSITIVE
10267,10268,synthetic-user-052,SRE-Role,autoscaling:ExitStandby,0,152,MEDIUM,5,INTENDED,0.859780,EXCESSIVE,FALSE_POSITIVE
6315,6316,synthetic-user-032,CloudEngineer-Role,route53:DeleteTrafficPolicyInstance,0,47,CRITICAL,10,INTENDED,0.852458,EXCESSIVE,FALSE_POSITIVE
7210,7211,synthetic-user-037,SecurityEngineer-Role,secretsmanager:TagResource,3,109,LOW,3,INTENDED,0.850819,EXCESSIVE,FALSE_POSITIVE
6346,6347,synthetic-user-032,CloudEngineer-Role,route53:DeleteHealthCheck,1,56,HIGH,8,INTENDED,0.849887,EXCESSIVE,FALSE_POSITIVE
215,216,synthetic-user-002,BackendDeveloper-Role,dynamodb:ListExports,0,112,LOW,2,INTENDED,0.840244,EXCESSIVE,FALSE_POSITIVE
350,351,synthetic-user-002,BackendDeveloper-Role,dynamodb:UpdateKinesisStreamingDestination,1,111,MEDIUM,5,INTENDED,0.830725,EXCESSIVE,FALSE_POSITIVE
3376,3377,synthetic-user-017,DataScientist-Role,comprehend:StopTrainingDocumentClassifier,1,61,MEDIUM,5,INTENDED,0.828837,EXCESSIVE,FALSE_POSITIVE
8340,8341,synthetic-user-042,IAMAdministrator-Role,sts:AssumeRole,2,33,CRITICAL,10,INTENDED,0.828226,EXCESSIVE,FALSE_POSITIVE


In [7]:
# Held-out validation permutation importance.
perm = permutation_importance(
    selected_estimator, X_val, y_val,
    scoring="average_precision", n_repeats=10, random_state=42, n_jobs=1
)
importance = pd.DataFrame({
    "feature": X_val.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)
display(importance.head(20))
importance.to_csv(OUTPUT_DIR / "selected_model_permutation_importance.csv", index=False)

,feature,importance_mean,importance_std
15,service,0.649057,0.008082
20,department,0.218118,0.010013
19,position,0.190358,0.004914
4,days_since_last_use,0.007820,0.001987
0,usage_count,0.003742,0.001399
21,team,0.001102,0.000137
10,scope_breadth,0.000629,0.000220
18,action_family,0.000376,0.000337
14,is_security_sensitive,0.000150,0.000038
11,is_wildcard_resource,0.000087,0.000084


In [8]:
# Robustness: missing values should not crash inference.
X_missing = X_val.copy()
for col in [c for c in ["usage_count", "days_since_last_use"] if c in X_missing.columns]:
    X_missing.loc[X_missing.index[:10], col] = np.nan
for col in models[selected_model_name]["categorical_features"]:
    if col in X_missing.columns:
        X_missing.loc[X_missing.index[:5], col] = np.nan

missing_prob = selected_estimator.predict_proba(X_missing)[:, 1]
print("Missing-value robustness finite:", bool(np.isfinite(missing_prob).all()))

# Robustness: unknown categorical levels should not crash the encoder.
X_unknown = X_val.copy()
for col in models[selected_model_name]["categorical_features"]:
    if col in X_unknown.columns:
        X_unknown[col] = "__UNSEEN_CATEGORY__"

unknown_prob = selected_estimator.predict_proba(X_unknown)[:, 1]
print("Unknown-category robustness finite:", bool(np.isfinite(unknown_prob).all()))

Missing-value robustness finite: True
Unknown-category robustness finite: True


In [9]:
selection = {
    "selected_model_name": selected_model_name,
    "selection_metric": "validation_pr_auc",
    "validation_pr_auc": float(comparison.loc[selected_model_name, "pr_auc"]),
    "validation_threshold_reference": float(validation_threshold)
}
with open(ARTIFACT_DIR / "model_selection.json", "w", encoding="utf-8") as f:
    json.dump(selection, f, indent=2)
print("Saved:", ARTIFACT_DIR / "model_selection.json")

Saved: C:\Users\LENOVO\Downloads\IAM_Model_V2_Notebooks\artifacts\model_selection.json
